In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

import koreanize_matplotlib

In [4]:
# 1~8호선 심도 데이터, 9호선 심도 데이터, 통합된 기본 데이터 가져오기
depth_1to8_df = pd.read_csv("../../data/raw/서울교통공사_1~8호선_역사심도정보_20250814.csv", encoding='cp949')
depth_9_df = pd.read_csv("../../data/raw/서울교통공사_9호선 2_3단계 역사 심도 정보_20250813.csv", encoding='cp949')
base_df = pd.read_excel("../../data/processed/team/subway_merged_base.xlsx")

In [5]:
# 1~8호선 필요 컬럼 추출 후 컬럼명 변환
depth_1to8_cleaned_df = depth_1to8_df[['역명', '층수', '정거장깊이']].copy()
depth_1to8_cleaned_df.rename(columns={'역명' : '지하철역', '정거장깊이' : '심도'}, inplace=True)

# 9호선 필요 컬럼 추출 후 컬럼명 변환
depth_9_cleaned_df = depth_9_df[['정거장명', '정거장층수', '승강장기준 정거장 깊이(m)']].copy()
depth_9_cleaned_df.rename(columns={'정거장명':'지하철역', '정거장층수':'층수', '승강장기준 정거장 깊이(m)':'심도'}, inplace=True)

In [6]:
# 1~8호선 필요 컬럼 추출 데이터와 9호선 필요 컬럼 추출 데이터를 행 방향으로 병합
depth_all_df = pd.concat([depth_1to8_cleaned_df, depth_9_cleaned_df], axis=0, ignore_index=True)

In [8]:
# 서울시가 아닌 역들 제외하기
non_seoul = [
    # 5호선 (하남)
    '미사', '하남검단산', '하남시청(덕풍?신장)', '하남풍산',
    # 7호선 (경기/인천)
    '광명사거리', '굴포천', '까치울', '부천시청', '부평구청',
    '삼산체육관', '상동', '신중동', '춘의', '철산',
    # 8호선 (성남)
    '남위례', '남한산성입구(성남법원.검찰청)', '단대오거리',
    '모란', '복정', '산성', '수진', '신흥',
]

depth_all_cleaned_df = depth_all_df[~depth_all_df['지하철역'].isin(non_seoul)].copy()
depth_all_cleaned_df

,지하철역,층수,심도
0,서울,B2,11.850
1,시청,B2,10.050
2,종각,B2,11.430
3,종로3가,B2,11.240
4,종로5가,B2,11.390
...,...,...,...
284,송파나루,B2,17.381
285,한성백제,B2,17.970
286,올림픽공원,B3,22.370
287,둔촌오륜,B2,21.410


In [10]:
# 맨 뒤에 역이라고 되어 있는 거는 역만 삭제
depth_all_cleaned_df['지하철역'] = depth_all_cleaned_df['지하철역'].str.replace('역$', '', regex=True)

# 앞 뒤 공백 제거
depth_all_cleaned_df['지하철역'] = depth_all_cleaned_df['지하철역'].str.strip()

In [11]:
# 지하철역별로 층수와 심도 집계 -> 지하철역이 겹칠 땐 더 깊은 역으로 집계
depth_final_df = depth_all_cleaned_df.groupby('지하철역').agg({
    '층수' : 'max',
    '심도' : 'max'
}).reset_index()

depth_final_df

,지하철역,층수,심도
0,DMC,B2,11.57
1,가락시장,B4,23.33
2,가산디지털단지,B4,26.67
3,강남,B2,11.96
4,강남구청,B3,21.72
...,...,...,...
237,홍제,B3,15.82
238,화곡,B3,18.90
239,화랑대,B2,15.57
240,회현,B4,22.09
